[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_12_logsumexp_solution.ipynb)

# 🟡 Solution: LogSumExp

*Core Ops & Layers · Medium*

Reference implementation. Try it yourself in `b_12_logsumexp.ipynb` first.

---
Implement **logsumexp** two ways: as a normal reduction, and as a *streaming*
merge that never sees all the data at once.

$$\text{logsumexp}(x) = \log \sum_i e^{x_i}
= m + \log \sum_i e^{x_i - m}, \qquad m = \max_i x_i$$

### Signatures
```python
def logsumexp(x, axis=-1, keepdims=False): ...
def logsumexp_merge(m1, l1, m2, l2): ...      # -> (m, l)
```

### Rules
- No `jax.scipy.special.logsumexp` and no `jax.nn.logsumexp`
- Stable for large positive **and** large negative inputs
- `axis` may be any valid axis; `keepdims` must behave like every other
  reduction

---

## Part 1 — the reduction

### The trap that makes this its own problem
Softmax uses the same max trick, so it is tempting to assume the same shape
handling carries over. It does not, and the difference is the point.

Softmax **divides** by its sum. With `keepdims=True` on both reductions the
shapes cancel, so uniform `keepdims` is simply correct:

```python
z = x - x.max(axis, keepdims=True)             # (2, 3)
e = jnp.exp(z) / jnp.sum(..., keepdims=True)   # (2, 3) / (2, 1) -> (2, 3)  ✓
```

logsumexp **adds** the max back. The sum has already reduced the axis away, so
the operands no longer match:

```python
x_max = jnp.max(x, axis=axis, keepdims=True)              # (2, 1)
s = jnp.log(jnp.sum(jnp.exp(x - x_max), axis=axis))       # (2,)
s + x_max     # (2,) + (2, 1) -> (2, 2)   WRONG, and it does not raise
```

That is an outer sum. On a `(2, 3)` input it silently returns `(2, 2)` with each
row's value duplicated. Nothing errors, and a test that only checks values at
`[0]` still passes. You need the max at the *reduced* rank —
`x_max.squeeze(axis)` — or `keepdims=True` on both and one squeeze at the end.

A square input hides this completely: `(3,) + (3, 1)` broadcasts to `(3, 3)`
without complaint. Test with something like `(2, 3)`.

### Why the max must be per-slice
Shifting by any constant is exact, so on a single vector a global `max(x)` also
works. Across slices on different scales it does not: subtract a global `1000`
from a row sitting near `-1000` and every term underflows to `0`, leaving
`log(0) = -inf`. Reduce along the axis you are reducing over.

---

## Part 2 — the streaming merge

Represent a partial result as the pair $(m, \ell)$ with
$m = \max x_i$ and $\ell = \sum_i e^{x_i - m}$, so the answer is
$m + \log \ell$. `logsumexp_merge` combines two such pairs:

$$m = \max(m_1, m_2), \qquad
\ell = \ell_1 e^{m_1 - m} + \ell_2 e^{m_2 - m}$$

Both sums are rescaled onto the *new* max before adding — that rescale is the
whole trick, and it is why the running total never overflows no matter what
order the chunks arrive in.

This is exactly FlashAttention's inner loop (problem 25): the running `m` and
`l` carried across key tiles, with everything accumulated so far retroactively
rescaled whenever a later tile raises the maximum. It is also why attention
never needs to materialise the full `(seq_q, seq_k)` score matrix.

### The empty state
The identity element is $(-\infty,\, 0)$ — merging it changes nothing, which is
what lets you start a fold from it. Watch the arithmetic: when *both* inputs are
$-\infty$ the new max is $-\infty$ too, and `exp(-inf - -inf)` is `nan`, not `0`.
Pin the exponent when the max is not finite.

### A useful identity
$$\frac{\partial}{\partial x_i}\,\text{logsumexp}(x) = \text{softmax}(x)_i$$

so the gradient is a probability distribution and sums to 1 — the fastest way to
check your implementation differentiates correctly.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def logsumexp(x, axis=-1, keepdims=False):
    # keepdims=True here so the max broadcasts against x for the subtraction.
    x_max = jnp.max(x, axis=axis, keepdims=True)
    # An all -inf slice has max -inf, and -inf - -inf is nan. Shift by 0 there.
    x_max = jnp.where(jnp.isfinite(x_max), x_max, 0.0)

    # Keep the axis on the sum too, so both operands still line up when the max
    # is added back. Adding a (..., 1) max to an already-reduced (...,) sum
    # broadcasts into an outer sum instead — silently, and with the wrong rank.
    out = jnp.log(jnp.sum(jnp.exp(x - x_max), axis=axis, keepdims=True)) + x_max

    return out if keepdims else jnp.squeeze(out, axis=axis)


def logsumexp_merge(m1, l1, m2, l2):
    m = jnp.maximum(m1, m2)
    # Same -inf guard: the empty state is (-inf, 0), and merging two empties
    # would otherwise compute exp(-inf - -inf) = nan instead of 0.
    safe = jnp.where(jnp.isfinite(m), m, 0.0)
    # Rescale BOTH sums onto the new max before adding — this is what keeps the
    # running total finite regardless of the order chunks arrive in.
    return m, l1 * jnp.exp(m1 - safe) + l2 * jnp.exp(m2 - safe)

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

x = jnp.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])   # (2, 3), deliberately not square

print("mine     :", logsumexp(x, axis=-1))
print("reference:", jax.scipy.special.logsumexp(x, axis=-1))

# Failure 1 — the max added back at the wrong rank. No error, wrong shape.
x_max = jnp.max(x, axis=-1, keepdims=True)          # (2, 1)
s = jnp.log(jnp.sum(jnp.exp(x - x_max), axis=-1))   # (2,)
print("\n(2,) + (2,1) ->", (s + x_max).shape, "  <- outer sum, silently wrong")
print(s + x_max)
print("squeezed     ->", (s + x_max.squeeze(-1)).shape, s + x_max.squeeze(-1))

# Failure 2 — a global max instead of a per-slice one.
rows = jnp.array([[1000.0, 1001.0, 1002.0], [-1000.0, -1001.0, -1002.0]])
m = jnp.max(rows)                                   # one number for both rows
print("\nglobal max :", jnp.log(jnp.sum(jnp.exp(rows - m), axis=-1)) + m)
print("per-slice  :", logsumexp(rows, axis=-1))

# Failure 3 — no shift at all.
big = jnp.array([1000.0, 1001.0, 1002.0])
print("\nnaive      :", jnp.log(jnp.sum(jnp.exp(big))))
print("stable     :", logsumexp(big))

# Streaming: fold over chunks, never holding them all at once.
chunks = [jnp.array([1.0, 2.0, 3.0]), jnp.array([100.0, 101.0]), jnp.array([-50.0])]
state = (-jnp.inf, 0.0)                             # the empty state
for c in chunks:
    cm = jnp.max(c)
    state = logsumexp_merge(*state, cm, jnp.sum(jnp.exp(c - cm)))
m_all, l_all = state
print("\nstreamed   :", m_all + jnp.log(l_all))
print("one-shot   :", logsumexp(jnp.concatenate(chunks)))

# The gradient is softmax — a correctness check that costs one line.
g = jax.grad(lambda v: logsumexp(v))(jnp.array([1.0, 2.0, 3.0]))
print("\ngrad       :", g)
print("softmax    :", jax.nn.softmax(jnp.array([1.0, 2.0, 3.0])))

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("logsumexp")